In [ ]:
#| default_exp parallel

In [ ]:
#| export
from fastcore.imports import *
from fastcore.basics import *
from fastcore.foundation import *
from fastcore.meta import *
from fastcore.xtras import *
from functools import wraps

import concurrent.futures,time
from multiprocessing import Process,Queue,Manager,set_start_method,get_all_start_methods,get_context
from threading import Thread,Lock
try:
    if sys.platform == 'darwin' and IN_NOTEBOOK: set_start_method("fork")
except: pass

In [ ]:
import asyncio
from fastcore.test import *
from nbdev.showdoc import *
from fastcore.nb_imports import *

# Parallel

> Threading and multiprocessing functions

In [ ]:
#| export
def threaded(
    process=False, # Create a Process instead of a Thread?
    daemon=False   # Use daemon mode?
): # The Process or Thread created, which will have `result` attr injected in once complete
    "Run `f` in a `Thread` (or `Process` if `process=True`), and returns it"
    def _r(f):
        def g(_obj_td, *args, **kwargs):
            res = f(*args, **kwargs)
            _obj_td.result = res
        @wraps(f)
        def _f(*args, **kwargs):
            Proc = get_context('fork').Process if sys.platform == 'darwin' else Process
            res = (Thread,Proc)[process](target=g, args=args, kwargs=kwargs)
            res._args = (res,)+res._args
            if daemon: res.daemon = True
            res.start()
            return res
        return _f
    if callable(process):
        o = process
        process = False
        return _r(o)
    return _r

In [ ]:
@threaded
def _1():
    time.sleep(0.05)
    print("second")
    return 5

@threaded
def _2():
    time.sleep(0.01)
    print("first")

a = _1()
_2()
time.sleep(0.1)

first
second


After the thread is complete, the return value is stored in the `result` attr.

In [ ]:
#| eval: false
a.result

5

Pass `daemon=True` to make the thread (or process) a daemon, so it won't prevent the parent from exiting. Useful for background services like webservers, where you don't want a still-running thread to block process shutdown.

In [ ]:
@threaded(daemon=True)
def f(): time.sleep(0.01)

assert f().daemon

In [ ]:
#| export
def startthread(f=None, daemon=False):
    "Like `threaded`, but start thread immediately"
    def _f(f): return threaded(daemon=daemon)(f)()
    return _f(f) if f else _f


In [ ]:
@startthread
def _():
    time.sleep(0.05)
    print("second")

@startthread
def _():
    time.sleep(0.01)
    print("first")

time.sleep(0.1)

first
second


In [ ]:
@startthread(daemon=True)
def f(): time.sleep(0.01)

assert f.daemon

In [ ]:
#| export
def startproc(f=None, daemon=False):
    "Like `threaded(True)`, but start Process immediately"
    def _f(f): return threaded(True, daemon=daemon)(f)()
    return _f(f) if f else _f


In [ ]:
@startproc
def _():
    time.sleep(0.05)
    print("second")

@startproc
def _():
    time.sleep(0.01)
    print("first")

time.sleep(0.1)

In [ ]:
#| export
def _call(lock, pause, n, g, item, return_exceptions=False):
    l = False
    if pause:
        try:
            l = lock.acquire(timeout=pause*(n+2))
            time.sleep(pause)
        finally:
            if l: lock.release()
    try: return g(item)
    except Exception as e:
        if return_exceptions: return e
        raise

In [ ]:
#| export
def parallelable(param_name, num_workers, f=None):
    f_in_main = f == None or sys.modules[f.__module__].__name__ == "__main__"
    if sys.platform == "win32" and IN_NOTEBOOK and num_workers > 0 and f_in_main:
        print("Due to IPython and Windows limitation, python multiprocessing isn't available now.")
        print(f"So `{param_name}` has to be changed to 0 to avoid getting stuck")
        return False
    return True

In [ ]:
#| export
class ThreadPoolExecutor(concurrent.futures.ThreadPoolExecutor):
    "Same as Python's ThreadPoolExecutor, except can pass `max_workers==0` for serial execution"
    def __init__(self, max_workers=defaults.cpus, on_exc=print, pause=0, **kwargs):
        if max_workers is None: max_workers=defaults.cpus
        store_attr()
        self.not_parallel = max_workers==0
        if self.not_parallel: max_workers=1
        super().__init__(max_workers, **kwargs)

    def map(self, f, items, *args, timeout=None, chunksize=1, return_exceptions=False, **kwargs):
        self.lock = Lock() if self.not_parallel==False and self.pause else None
        g = partial(f, *args, **kwargs)
        if self.not_parallel: return map(g, items)
        _g = partial(_call, self.lock, self.pause, self.max_workers, g, return_exceptions=return_exceptions)
        try: return super().map(_g, items, timeout=timeout, chunksize=chunksize)
        except Exception as e: self.on_exc(e)

In [ ]:
show_doc(ThreadPoolExecutor, title_level=4)

<div class="prose" markdown="1">

---

[source](https://github.com/AnswerDotAI/fastcore/blob/main/fastcore/parallel.py#L89){target="_blank" style="float:right; font-size:smaller"}

#### ThreadPoolExecutor

```python
def ThreadPoolExecutor(
    max_workers:int=16, on_exc:builtin_function_or_method=print, pause:int=0, **kwargs
):
```

*Same as Python's ThreadPoolExecutor, except can pass `max_workers==0` for serial execution*

</div>

In [ ]:
#| export
@delegates()
class ProcessPoolExecutor(concurrent.futures.ProcessPoolExecutor):
    "Same as Python's ProcessPoolExecutor, except can pass `max_workers==0` for serial execution"
    def __init__(self, max_workers=defaults.cpus, on_exc=print, pause=0, **kwargs):
        if max_workers is None: max_workers=defaults.cpus
        store_attr()
        self.not_parallel = max_workers==0
        if self.not_parallel: max_workers=1
        super().__init__(max_workers, **kwargs)

    def map(self, f, items, *args, timeout=None, chunksize=1, return_exceptions=False, **kwargs):
        if not parallelable('max_workers', self.max_workers, f): self.max_workers = 0
        self.not_parallel = self.max_workers==0
        if self.not_parallel: self.max_workers=1

        self.lock = Manager().Lock() if self.not_parallel==False and self.pause else None
        g = partial(f, *args, **kwargs)
        if self.not_parallel: return map(g, items)
        _g = partial(_call, self.lock, self.pause, self.max_workers, g, return_exceptions=return_exceptions)
        try: return super().map(_g, items, timeout=timeout, chunksize=chunksize)
        except Exception as e: self.on_exc(e)

In [ ]:
show_doc(ProcessPoolExecutor, title_level=4)

<div class="prose" markdown="1">

---

[source](https://github.com/AnswerDotAI/fastcore/blob/main/fastcore/parallel.py#L108){target="_blank" style="float:right; font-size:smaller"}

#### ProcessPoolExecutor

```python
def ProcessPoolExecutor(
    max_workers:int=16, on_exc:builtin_function_or_method=print, pause:int=0, mp_context:NoneType=None,
    initializer:NoneType=None, initargs:tuple=(), max_tasks_per_child:NoneType=None
):
```

*Same as Python's ProcessPoolExecutor, except can pass `max_workers==0` for serial execution*

</div>

In [ ]:
#| export
def parallel(f, items, *args, n_workers=defaults.cpus, total=None, progress=None, pause=0,
             method=None, threadpool=False, timeout=None, chunksize=1, return_exceptions=False, **kwargs):
    "Applies `func` in parallel to `items`, using `n_workers`"
    kwpool = {}
    if threadpool: pool = ThreadPoolExecutor
    else:
        if not method and sys.platform == 'darwin': method='fork'
        if method: kwpool['mp_context'] = get_context(method)
        pool = ProcessPoolExecutor
    with pool(n_workers, pause=pause, **kwpool) as ex:
        r = ex.map(f,items, *args, timeout=timeout, chunksize=chunksize, return_exceptions=return_exceptions, **kwargs)
        if progress:
            from fastprogress import progress_bar
            if total is None: total = len(items)
            r = progress_bar(r, total=total, leave=False)
        return L(r)

In [ ]:
#| export
def _add_one(x, a=1):
    # this import is necessary for multiprocessing in notebook on windows
    import random
    time.sleep(random.random()/80)
    return x+a

In [ ]:
inp,exp = range(50),range(1,51)

test_eq(parallel(_add_one, inp, n_workers=2), exp)
test_eq(parallel(_add_one, inp, threadpool=True, n_workers=2), exp)
test_eq(parallel(_add_one, inp, n_workers=1, a=2), range(2,52))
test_eq(parallel(_add_one, inp, n_workers=0), exp)
test_eq(parallel(_add_one, inp, n_workers=0, a=2), range(2,52))

Use the `pause` parameter to ensure a pause of `pause` seconds between processes starting. This is in case there are race conditions in starting some process, or to stagger the time each process starts, for example when making many requests to a webserver. Set `threadpool=True` to use `ThreadPoolExecutor` instead of `ProcessPoolExecutor`.

In [ ]:
from datetime import datetime

In [ ]:
def print_time(i): 
    time.sleep(random.random()/1000)
    print(i, datetime.now())

parallel(print_time, range(5), n_workers=2, pause=0.1);

You can also pass `return_exceptions=True` to catch any exceptions from parallel workers and return them instead:

In [ ]:
def die_sometimes(x):
    if 3<x<6: raise Exception(f"exc: {x}")
    return x*2

parallel(die_sometimes, range(8), return_exceptions=True)

[0, 2, 4, 6, Exception('exc: 4'), Exception('exc: 5'), 12, 14]

In [ ]:
#| export
async def parallel_async(f, items, *args, n_workers=16, pause=0,
        timeout=None, chunksize=1, cancel_on_error=False, return_exceptions=False, **kwargs):
    "Applies `f` to `items` in parallel using asyncio and a semaphore to limit concurrency."
    import asyncio
    semaphore = asyncio.Semaphore(n_workers)
    async def limited_task(i, item):
        if pause: await asyncio.sleep(i * pause)
        async with semaphore:
            coro = f(item, *args, **kwargs) if asyncio.iscoroutinefunction(f) else asyncio.to_thread(f, item, *args, **kwargs)
            return await asyncio.wait_for(coro, timeout) if timeout else await coro
    if cancel_on_error:
        tasks = [asyncio.ensure_future(limited_task(i, item)) for i,item in enumerate(items)]
        try: return await asyncio.gather(*tasks)
        except BaseException:
            for t in tasks: t.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            raise
    return await asyncio.gather(*[limited_task(i, item) for i,item in enumerate(items)], return_exceptions=return_exceptions)

In [ ]:
async def print_time_async(i): 
    start =datetime.now()
    wait = random.random()/30
    await asyncio.sleep(wait)
    print(i, start, datetime.now(), wait)
    if i==5: raise Exception(f"exc {i}")
    return i

res = await parallel_async(print_time_async, range(6), n_workers=3, return_exceptions=True)
test_eq(res[:5], [0, 1, 2, 3, 4])
test_eq(type(res[5]), Exception)

2 2026-07-17 17:39:11.566435 2026-07-17 17:39:11.569794 0.002859891538949448
3 2026-07-17 17:39:11.569932 2026-07-17 17:39:11.576193 0.005425981904664934
1 2026-07-17 17:39:11.566330 2026-07-17 17:39:11.588110 0.020615564297547868
0 2026-07-17 17:39:11.566115 2026-07-17 17:39:11.596509 0.029235003044882336
4 2026-07-17 17:39:11.576326 2026-07-17 17:39:11.606287 0.02879390679028037
5 2026-07-17 17:39:11.588278 2026-07-17 17:39:11.606438 0.017552992195368936


Adding `pause` ensures a gap between starts:

In [ ]:
await parallel_async(print_time_async, range(6), n_workers=3, pause=0.1, return_exceptions=True);

With `cancel_on_error=True`, the first failure cancels all remaining tasks, waits for them to finish cancelling, then re-raises that first exception. (Earlier versions raised an `ExceptionGroup` here; now you catch the original exception directly.)

In [ ]:
async def maybe_fail(i:int):
    "Double i unless it's 3, in which case fail"
    await asyncio.sleep(random.random()/50)
    if i==3: raise ValueError(f"bad: {i}")
    return i*2

In [ ]:
with expect_fail(ValueError, contains='bad: 3'): await parallel_async(maybe_fail, range(6), n_workers=3, cancel_on_error=True)

With `return_exceptions=False`, an exception is raised on error:

In [ ]:
with expect_fail(ValueError): await parallel_async(maybe_fail, range(6), n_workers=3)

In [ ]:
#| export
def bg_task(coro):
    "Like `asyncio.create_task` but logs exceptions for fire-and-forget tasks"
    import traceback,asyncio
    def _done(t):
        if not t.cancelled() and (exc := t.exception()): traceback.print_exception(exc)
    task = asyncio.create_task(coro)
    task.add_done_callback(_done)
    return task

In [ ]:
async def _ok(): return 42
async def _fail(): raise ValueError("this error will be printed")

t1 = bg_task(_ok())
t2 = bg_task(_fail())
await asyncio.sleep(0.01)
test_eq(t1.result(), 42)

Traceback (most recent call last):
  File "<ipython-input-1-48a55f4f8ca9>", line 2, in _fail
    async def _fail(): raise ValueError("this error will be printed")
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: this error will be printed


# Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()